<h1>LD Pipeline 2024 (Version 1.0)</h1>

<h3>Python-Bibliotheken laden</h3>

In [1]:
env = "INT"

In [2]:
import os
import requests

try:
    initialized
except NameError:
    initialized = False

    # if not initialized:
    initialized = True
    os.chdir("../")
    import main
    from pipeline.base import Utils

    Utils.is_jupyter_notebook = True
    if env == "INT":
        env = main.Env.int
    elif env == "PROD":
        env = main.Env.prod
    print("hallo")

hallo


<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    1
</span> Überprüfe des Start-Signals</h3>
<p style="max-width: 700px;">
    Es wird überprüft, ob im Ordner <b>\\\szh.loc\ssz\applikationen\HDB_Dropzone\PROD\Test\Pipeline</b> ein Start-Signal vorhanden ist. Falls ja, wird das Start-Signal in ein Running-Signal umgewandelt und die Pipeline beginnt mit der Ausführung.
</p>

In [3]:
utils = Utils()
from pipeline.base import Environment
import main

environment = Environment(env)
print(environment)
print(os.getcwd())
print(environment.config.get("start_signal_folder"))

print(environment.config.get("logger.filepath"))

if utils.check_start_signal(env):
    print("Start signal found and renamed to a running signal.")
else:
    print("No start signal found.")

g:\sszads\ld-pipeline-2024
//szh.loc/ssz/applikationen/HDB_Dropzone/INT/Test/Pipeline
logs
No start signal found.


<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    2
</span> Aktualisiere die Pipe-Tabellen</h3>
<p>
    Die relevanten Daten werden aus der HDB in Tabellen mit dem Prefix "pipe_" kopiert, um sie in einen konsistenten Zustand zu archivieren.
</p>

In [4]:
main.step(name="copyHDBToPipeTables", env=env)

2025-08-08 11:19:40 - Copying HDB data to the "pipe" tables ...
2025-08-08 11:19:41 - Copying HDBCodeliste to pipe_HDBCodeliste ...

                                INSERT INTO pipe_HDBCodeliste
                                SELECT "CODE", "CODENAME", "REFERENZTABELLE", "Index" FROM #pipe_HDBCodeliste
                            
2025-08-08 11:19:46 - Done
2025-08-08 11:19:46 - Copying HDBCubeDefinition to pipe_HDBCubeDefinition ...

                                INSERT INTO pipe_HDBCubeDefinition
                                SELECT "CID", "Titel" FROM #pipe_HDBCubeDefinition
                            
2025-08-08 11:19:46 - Done
2025-08-08 11:19:46 - Copying HDBDatenattribute_TEST to pipe_HDBDatenattribute_TEST ...

                                INSERT INTO pipe_HDBDatenattribute_TEST
                                SELECT "id", "SASA_OutputsId", "Title", "Feldbeschreibung", "Sprechender Feldname", "Technischer Feldname" FROM #pipe_HDBDatenattribute_TEST
                    

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    3
</span> Generiere die Triple-Dateien</h3>

In [ ]:
triple_types_metadata = [
    "code",
    "cube",
    "groupCode",
    "hierarchy",
    "legalFoundation",
    "measureUnit",
    "measure",
    "property",
    "room",
    "time",
]
triple_types_observations = ["observation"]
triple_types_others = ["copyStatic", "buildTermsetHierarchy", "generateViews"]

# just for testing
# triple_types_metadata = ['groupCode']
# triple_types_observations = []
# triple_types_others = []


options_batching = {
    "db_batch_size": 100000,
    "write_batch_size": 600000,
    "max_iteration": None,
}

for name in triple_types_metadata:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_observations:
    main.step(name=f"{name}Templating", env=env, options=options_batching)
for name in triple_types_others:
    main.step(name=name, env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    4
</span> Setze das Start-Signal für die Generierung der Fuseki-Indexdateien</h3>

In [ ]:
utils = Utils()
utils.set_start_signal_fuseki_index(env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    5
</span> Schreibe die Publikationsstati zurück in die HDB</h3>

In [ ]:
main.step(name="writePublicationStatiToHDB", env=env)

<h3><span style="
    display: inline-block;
    width: 50px;
    height: 50px;
    border-radius: 50%;
    border: 3px solid black;
    font-size: 24px;
    font-weight: bold;
    text-align: center;
    line-height: 50px;
    background-color: yellow;
">
    6
</span> Setze das Running-Signal auf Finish</h3>

In [ ]:
utils = Utils()
utils.set_finish_signal(env)